# ABSA Laptop — Clean Ablation Training

Notebook này huấn luyện lại pipeline ABSA hiện tại trên dataset **Laptop**, giữ nguyên logic tuning, ablation, early stopping, refit và đánh giá như thực nghiệm Restaurant.

Trước khi chạy:

1. Upload `absa_laptop_ablation_fresh.zip` thành Kaggle Dataset.
2. Gắn Dataset đó vào notebook.
3. Bật GPU và Internet.
4. Chạy toàn bộ cell theo thứ tự.


In [1]:
from pathlib import Path
import os, shutil, zipfile, json, subprocess, sys

print('Python:', sys.version)
print('Kaggle working exists:', Path('/kaggle/working').exists())
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found; GPU may be disabled.')


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Kaggle working exists: True
Sun Jun 21 15:05:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |          

## 1. Giải nén package và tạo trạng thái clean-start

Mọi output của lần chạy cũ trong `/kaggle/working` được xóa trước khi bắt đầu.

In [2]:
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
INPUT = Path('/kaggle/input/datasets/banhbaogachcua/nlp-lap-update')
PROJECT = WORK / 'absa_laptop_ablation_fresh'

CORE_TUNING = WORK / 'tuning_laptop_ablation_core'
CORE_FINAL = WORK / 'outputs_laptop_ablation_core'
DESC_TUNING = WORK / 'tuning_laptop_ablation_desc'
DESC_FINAL = WORK / 'outputs_laptop_ablation_desc'
RESULT_ZIP = WORK / 'absa_laptop_results_fresh.zip'

CLEAN_RUN = True
if CLEAN_RUN:
    for path in [PROJECT, CORE_TUNING, CORE_FINAL, DESC_TUNING, DESC_FINAL, RESULT_ZIP]:
        if path.exists():
            print('Removing old:', path)
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()

if not INPUT.exists():
    raise FileNotFoundError(f'Không tìm thấy Kaggle Input: {INPUT}')

zip_candidates = list(INPUT.rglob('absa_laptop_ablation_fresh.zip'))
if not zip_candidates:
    zip_candidates = list(INPUT.rglob('*laptop*ablation*.zip'))

if zip_candidates:
    zip_path = zip_candidates[0]
    print('Extracting ZIP:', zip_path)
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(PROJECT)
else:
    # Kaggle thường tự giải nén file ZIP khi tạo Dataset. Khi đó copy trực tiếp
    # thư mục chứa train.py thay vì yêu cầu file ZIP còn tồn tại.
    train_candidates = list(INPUT.rglob('train.py'))
    if not train_candidates:
        visible = [str(path.relative_to(INPUT)) for path in INPUT.rglob('*') if path.is_file()][:50]
        raise FileNotFoundError(
            'Không tìm thấy ZIP hoặc train.py trong Kaggle Input. '
            f'Các file nhìn thấy: {visible}'
        )
    source_dir = train_candidates[0].parent
    print('Copying extracted source:', source_dir)
    shutil.copytree(source_dir, PROJECT, dirs_exist_ok=True)

if not (PROJECT / 'train.py').exists():
    candidates = list(PROJECT.rglob('train.py'))
    if not candidates:
        raise FileNotFoundError('Đã giải nén nhưng không tìm thấy train.py')
    PROJECT = candidates[0].parent

print('PROJECT =', PROJECT)
print('Files:', sorted(p.name for p in PROJECT.iterdir()))


Copying extracted source: /kaggle/input/datasets/banhbaogachcua/nlp-lap-update
PROJECT = /kaggle/working/absa_laptop_ablation_fresh
Files: ['CLEAN_START.md', 'PATCH_NOTES.md', 'absa_pipeline.py', 'dataset', 'kaggle_laptop_ablation_fresh.ipynb', 'requirements.txt', 'train.py', 'tune.py']


## 2. Cài đặt thư viện

In [3]:
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

## 3. Kiểm tra package và dataset Laptop

In [4]:
os.chdir(PROJECT)
required = [
    Path('train.py'), Path('tune.py'), Path('absa_pipeline.py'),
    Path('dataset/lap/train.multiple.json'),
    Path('dataset/lap/test.multiple.json'),
    Path('dataset/lap/aspects.json'),
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Thiếu file bắt buộc: {missing}'

forbidden_suffixes = {'.pt', '.pth', '.bin', '.safetensors'}
forbidden_dirs = {'outputs', 'runs', 'logs', 'wandb', '__pycache__'}
history_files = [
    str(path.relative_to(PROJECT)) for path in PROJECT.rglob('*')
    if path.suffix in forbidden_suffixes or any(part in forbidden_dirs for part in path.parts)
]
assert not history_files, f'Package chứa checkpoint/output cũ: {history_files[:20]}'

from collections import Counter
for split in ['train', 'test']:
    rows = json.loads(Path(f'dataset/lap/{split}.multiple.json').read_text(encoding='utf-8'))
    aspects = [a for row in rows for a in row.get('aspects', []) if a.get('polarity') in {'negative','neutral','positive'}]
    print(split, 'sentences=', len(rows), 'aspect_samples=', len(aspects), 'labels=', dict(Counter(a['polarity'] for a in aspects)))

subprocess.run([sys.executable, 'tune.py', '--help'], check=False)


train sentences= 1466 aspect_samples= 2328 labels= {'neutral': 464, 'positive': 994, 'negative': 870}
test sentences= 411 aspect_samples= 638 labels= {'positive': 341, 'negative': 128, 'neutral': 169}
usage: tune.py [-h] [--dataset {lap,rest,twi}] [--model MODEL]
               [--epochs EPOCHS] [--final-epochs FINAL_EPOCHS]
               [--patience PATIENCE] [--batch-size BATCH_SIZE]
               [--gradient-accumulation-steps GRADIENT_ACCUMULATION_STEPS]
               [--weight-decay WEIGHT_DECAY] [--valid-ratio VALID_RATIO]
               [--split-seed SPLIT_SEED] [--tuning-seeds TUNING_SEEDS]
               [--validation-seed-pairs VALIDATION_SEED_PAIRS]
               [--final-seeds FINAL_SEEDS] [--output-dir OUTPUT_DIR]
               [--final-output-dir FINAL_OUTPUT_DIR] [--amp]
               [--gradient-checkpointing] [--run-final]
               [--keep-trial-checkpoints] [--resume]
               [--search-space {ablation_all,ablation_core,ablation_desc,initial,refined}

CompletedProcess(args=['/usr/bin/python3', 'tune.py', '--help'], returncode=0)

## 4. Chạy core ablation trên Laptop

Giữ nguyên sáu trial và hyperparameter như thực nghiệm Restaurant. Tổng cộng có 18 validation run và 3 final run.

Nếu GPU hết bộ nhớ, đặt `BATCH_SIZE = 4` và `GRAD_ACCUM = 4`.

In [5]:
os.chdir(PROJECT)

VALID_RATIO = 0.15
BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 12
FINAL_EPOCHS = 15
SEED_PAIRS = '2024:2024,42:42,3407:3407'
FINAL_SEEDS = '2024,42,3407'

cmd = [
    sys.executable, 'tune.py',
    '--dataset', 'lap',
    '--search-space', 'ablation_core',
    '--validation-seed-pairs', SEED_PAIRS,
    '--valid-ratio', str(VALID_RATIO),
    '--final-seeds', FINAL_SEEDS,
    '--epochs', str(EPOCHS),
    '--final-epochs', str(FINAL_EPOCHS),
    '--patience', '3',
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRAD_ACCUM),
    '--amp',
    '--gradient-checkpointing',
    '--run-final',
    '--output-dir', str(CORE_TUNING),
    '--final-output-dir', str(CORE_FINAL),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


/usr/bin/python3 tune.py --dataset lap --search-space ablation_core --validation-seed-pairs 2024:2024,42:42,3407:3407 --valid-ratio 0.15 --final-seeds 2024,42,3407 --epochs 12 --final-epochs 15 --patience 3 --batch-size 8 --gradient-accumulation-steps 2 --amp --gradient-checkpointing --run-final --output-dir /kaggle/working/tuning_laptop_ablation_core --final-output-dir /kaggle/working/outputs_laptop_ablation_core
RUN: /usr/bin/python3 train.py --dataset lap --model microsoft/deberta-v3-base --epochs 12 --patience 3 --batch-size 8 --gradient-accumulation-steps 2 --weight-decay 0.01 --valid-ratio 0.15 --split-seed 2024 --seed 2024 --output-dir /kaggle/working/tuning_laptop_ablation_core --run-name tune_baseline_cls_split2024_seed2024 --evaluation-split validation --pooling-strategy cls --input-format term --encoder-layer-pooling last --encoder-learning-rate 8e-06 --head-learning-rate 2e-05 --dropout 0.3 --class-weighting none --neutral-weight-multiplier 1.0 --label-smoothing 0.02 --loss

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-21 15:06:09.550930: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782054369.947191      63 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782054370.052579      63 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 1978,
      "labels": {
        "negative": 743,
        "neutral": 393,
        "positive": 842
      },
      "unique_aspects": 844
    },
    "train_all": {
      "samples": 2327,
      "labels": {
        "negative": 870,
        "neutral": 463,
        "positive": 994
      },
      "unique_aspects": 949,
      "unseen_aspect_samples": 112,
      "unseen_aspect_ratio": 0.0481
    },
    "valid": {
      "samples": 349,
      "labels": {
        "negative": 127,
        "neutral": 70,
        "positive": 152
      },
      "unique_aspects": 212,
      "unseen_aspect_samples": 112,
      "unseen_aspect_ratio": 0.3209
    },
    "test": {
      "samples": 638,
      "labels": {
        "negative": 128,
        "neutral": 169,
        "positive": 341
      },
      "unique_aspects": 389,
      "unseen_aspect_samples": 283,
      "unseen_aspect_ratio": 0.4436
    },
    "e

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-21 15:15:51.417708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782054951.437312     108 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782054951.443846     108 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 1976,
      "labels": {
        "negative": 739,
        "neutral": 392,
        "positive": 845
      },
      "unique_aspects": 845
    },
    "train_all": {
      "samples": 2327,
      "labels": {
        "negative": 870,
        "neutral": 463,
        "positive": 994
      },
      "unique_aspects": 949,
      "unseen_aspect_samples": 105,
      "unseen_aspect_ratio": 0.0451
    },
    "valid": {
      "samples": 351,
      "labels": {
        "negative": 131,
        "neutral": 71,
        "positive": 149
      },
      "unique_aspects": 220,
      "unseen_aspect_samples": 105,
      "unseen_aspect_ratio": 0.2991
    },
    "test": {
      "samples": 638,
      "labels": {
        "negative": 128,
        "neutral": 169,
        "positive": 341
      },
      "unique_aspects": 389,
      "unseen_aspect_samples": 278,
      "unseen_aspect_ratio": 0.4357
    },
    "e

...

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-21 18:02:03.615531: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782064923.634758     595 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782064923.641196     595 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 1978,
      "labels": {
        "negative": 743,
        "neutral": 393,
        "positive": 842
      },
      "unique_aspects": 844
    },
    "train_all": {
      "samples": 2327,
      "labels": {
        "negative": 870,
        "neutral": 463,
        "positive": 994
      },
      "unique_aspects": 949,
      "unseen_aspect_samples": 112,
      "unseen_aspect_ratio": 0.0481
    },
    "valid": {
      "samples": 349,
      "labels": {
        "negative": 127,
        "neutral": 70,
        "positive": 152
      },
      "unique_aspects": 212,
      "unseen_aspect_samples": 112,
      "unseen_aspect_ratio": 0.3209
    },
    "test": {
      "samples": 638,
      "labels": {
        "negative": 128,
        "neutral": 169,
        "positive": 341
      },
      "unique_aspects": 389,
      "unseen_aspect_samples": 283,
      "unseen_aspect_ratio": 0.4436
    },
    "e

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-06-21 18:23:50.962998: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782066230.982545     623 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782066230.988728     623 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

pipeline_version=absa-training-v4-clean-ablation
{
  "device": "cuda",
  "data": {
    "train": {
      "samples": 1978,
      "labels": {
        "negative": 743,
        "neutral": 393,
        "positive": 842
      },
      "unique_aspects": 844
    },
    "train_all": {
      "samples": 2327,
      "labels": {
        "negative": 870,
        "neutral": 463,
        "positive": 994
      },
      "unique_aspects": 949,
      "unseen_aspect_samples": 112,
      "unseen_aspect_ratio": 0.0481
    },
    "valid": {
      "samples": 349,
      "labels": {
        "negative": 127,
        "neutral": 70,
        "positive": 152
      },
      "unique_aspects": 212,
      "unseen_aspect_samples": 112,
      "unseen_aspect_ratio": 0.3209
    },
    "test": {
      "samples": 638,
      "labels": {
        "negative": 128,
        "neutral": 169,
        "positive": 341
      },
      "unique_aspects": 389,
      "unseen_aspect_samples": 283,
      "unseen_aspect_ratio": 0.4436
    },
    "e

CompletedProcess(args=['/usr/bin/python3', 'tune.py', '--dataset', 'lap', '--search-space', 'ablation_core', '--validation-seed-pairs', '2024:2024,42:42,3407:3407', '--valid-ratio', '0.15', '--final-seeds', '2024,42,3407', '--epochs', '12', '--final-epochs', '15', '--patience', '3', '--batch-size', '8', '--gradient-accumulation-steps', '2', '--amp', '--gradient-checkpointing', '--run-final', '--output-dir', '/kaggle/working/tuning_laptop_ablation_core', '--final-output-dir', '/kaggle/working/outputs_laptop_ablation_core'], returncode=0)

## 5. Tổng hợp kết quả core

In [6]:
import pandas as pd

summary_path = CORE_TUNING / 'tuning_summary.csv'
final_path = CORE_FINAL / 'final_runs_summary.csv'
meanstd_path = CORE_FINAL / 'final_metrics_mean_std.json'

summary = pd.read_csv(summary_path)
cols = [c for c in ['trial','mean_macro_f1','std_macro_f1','robust_macro_f1','mean_accuracy','mean_ece','pooling_strategy','input_format','neutral_weight_multiplier'] if c in summary.columns]
display(summary[cols])

if final_path.exists():
    display(pd.read_csv(final_path))
if meanstd_path.exists():
    print(json.dumps(json.loads(meanstd_path.read_text()), indent=2))


,trial,mean_macro_f1,std_macro_f1,robust_macro_f1,mean_accuracy,mean_ece,pooling_strategy,input_format,neutral_weight_multiplier
0,aspect_attention_term,0.780567,0.005838,0.774729,0.816400,0.052033,aspect_attention,term,1.00
1,aspect_attention_neutral125,0.781467,0.010403,0.771064,0.810667,0.056467,aspect_attention,term,1.25
2,hybrid_term,0.775233,0.018891,0.756343,0.817300,0.047533,hybrid,term,1.00
3,baseline_cls,0.777500,0.023473,0.754027,0.819200,0.045800,cls,term,1.00
4,cls_neutral150,0.772033,0.023350,0.748683,0.808733,0.036833,cls,term,1.50
5,cls_neutral125,0.767767,0.021833,0.745934,0.814467,0.051167,cls,term,1.25


,seed,best_epoch,accuracy,precision_macro,recall_macro,macro_f1,weighted_f1,roc_auc_macro,log_loss,expected_calibration_error,temperature
0,2024,5,0.7837,0.7592,0.7476,0.7224,0.7630,0.9132,0.5753,0.0785,1.614072
1,42,10,0.8150,0.7778,0.7909,0.7754,0.8111,0.9171,0.5389,0.0496,1.861002
2,3407,5,0.8072,0.7869,0.7693,0.7475,0.7900,0.9175,0.5677,0.0694,1.646387


{
  "accuracy": {
    "mean": 0.801967,
    "std": 0.013303
  },
  "precision_macro": {
    "mean": 0.774633,
    "std": 0.011528
  },
  "recall_macro": {
    "mean": 0.769267,
    "std": 0.017677
  },
  "macro_f1": {
    "mean": 0.748433,
    "std": 0.021647
  },
  "weighted_f1": {
    "mean": 0.788033,
    "std": 0.019686
  },
  "roc_auc_macro": {
    "mean": 0.915933,
    "std": 0.00194
  },
  "log_loss": {
    "mean": 0.560633,
    "std": 0.015678
  },
  "expected_calibration_error": {
    "mean": 0.065833,
    "std": 0.012065
  }
}


## 6. Tùy chọn: chạy aspect-description ablation

Mặc định tắt để không trộn kết quả core với tác động của `aspects.json`.

In [7]:
RUN_DESC_ABLATION = False

if RUN_DESC_ABLATION:
    desc_cmd = [
        sys.executable, 'tune.py',
        '--dataset', 'lap',
        '--search-space', 'ablation_desc',
        '--validation-seed-pairs', SEED_PAIRS,
        '--valid-ratio', str(VALID_RATIO),
        '--final-seeds', FINAL_SEEDS,
        '--epochs', str(EPOCHS),
        '--final-epochs', str(FINAL_EPOCHS),
        '--patience', '3',
        '--batch-size', str(BATCH_SIZE),
        '--gradient-accumulation-steps', str(GRAD_ACCUM),
        '--amp', '--gradient-checkpointing', '--run-final',
        '--output-dir', str(DESC_TUNING),
        '--final-output-dir', str(DESC_FINAL),
    ]
    print(' '.join(desc_cmd))
    subprocess.run(desc_cmd, check=True)
else:
    print('Description ablation is disabled.')


Description ablation is disabled.


## 7. Đóng gói kết quả để tải về

Checkpoint và các ZIP artifact lồng nhau được bỏ qua mặc định để tránh file kết quả tăng lên hàng GB. Các JSON, CSV, prediction và biểu đồ vẫn được giữ.

In [8]:
INCLUDE_CHECKPOINTS = False
folders = [CORE_TUNING, CORE_FINAL, DESC_TUNING, DESC_FINAL]

with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in folders:
        if not folder.exists():
            continue
        for path in folder.rglob('*'):
            if not path.is_file():
                continue
            if not INCLUDE_CHECKPOINTS and path.suffix in {'.pt', '.pth', '.bin', '.safetensors'}:
                continue
            if path.name.endswith('_artifacts.zip'):
                continue
            zf.write(path, path.relative_to(WORK))

print('Created:', RESULT_ZIP)
print('Size MB:', round(RESULT_ZIP.stat().st_size / 1024 / 1024, 2))


Created: /kaggle/working/absa_laptop_results_fresh.zip
Size MB: 8.94
